# Notebook 03 - Preprocessing

Turning the raw data into something the models can use. This runs the DS1
pipeline end to end (impute, encode, scale, split), then preps the DS3
regression data and the DS2/DS4 validation sets. Anything learned along the
way (medians, the scaler, the one-hot categories) is fit on the training
split only, so the test split stays honest.



## Setup

Import the preprocessing module and load the raw primary dataset. The
pipeline reloads data through the validated loader itself, so the frame here
is just for showing the steps.

In [1]:
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ml" / "config.py").exists():
            return candidate
    raise RuntimeError(f"Could not locate the project root from {start}")


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml import config
from ml.preprocessing import (
    BehavioralPreprocessor,
    derive_procrastination_level,
    prepare_primary_data,
    prepare_regression_data,
    prepare_validation_set,
)
from data.data_loader import load_dataset

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("preprocessing")

raw = load_dataset("behavioral_analytics")
logger.info("Raw DS1 loaded: %d rows x %d columns", *raw.shape)

2026-07-03 18:32:01,542 | INFO | Loaded behavioral_analytics: 1200 rows x 36 columns from behavioral_analytics.csv


2026-07-03 18:32:01,543 | INFO | behavioral_analytics passed schema validation.


2026-07-03 18:32:01,543 | INFO | Raw DS1 loaded: 1200 rows x 36 columns


## Missing values

About a dozen behavioural columns are 2-5% empty. Dropping rows over that
would throw away too many partly-complete students, so each column gets its
training-split median instead. The nominal fields and the procrastination
source columns have no gaps.

In [2]:
null_report = (
    raw.isnull().sum()[lambda s: s > 0].sort_values(ascending=False).rename("nulls")
)
null_report.to_frame().assign(null_pct=(null_report / len(raw) * 100).round(2))

,nulls,null_pct
events_participation,58,4.83
sleep_hours,55,4.58
external_resources,45,3.75
focus_duration,43,3.58
routine_rating,41,3.42
programming_foundation,41,3.42
online_courses,36,3.00
stress_level,35,2.92
daily_productivity,34,2.83
revision_frequency,34,2.83


## The procrastination feature

DS1 has no procrastination column, so I build one from three behaviours:
internal_barrier, study_consistency, and tasks_on_time. The component scores
in config get summed and binned into Low/Medium/High. Here it is for a few
students, next to the source values it came from.

In [3]:
procrastination = derive_procrastination_level(raw)

sample = raw[config.PROCRASTINATION_SOURCE_COLUMNS].copy()
sample["-> procrastination_level"] = procrastination
display(sample.head(8))

logger.info("Procrastination distribution: %s", procrastination.value_counts().to_dict())

,internal_barrier,study_consistency,tasks_on_time,-> procrastination_level
0,Lack of Consistency or Determination (Difficul...,Rarely,Sometimes,High
1,Lack of Consistency or Determination (Difficul...,Sometimes,Sometimes,High
2,Difficulty with Focus / Concentration,Mostly consistent,Often,Low
3,Procrastination / Low Motivation,Sometimes,Often,Medium
4,Lack of Consistency or Determination (Difficul...,Sometimes,Sometimes,High
5,Procrastination / Low Motivation,Mostly consistent,Often,Medium
6,Difficulty with Focus / Concentration,Sometimes,Always,Medium
7,Poor Time Management / Over-scheduling,Mostly consistent,Often,Low


2026-07-03 18:32:01,557 | INFO | Procrastination distribution: {'Medium': 566, 'High': 499, 'Low': 135}


## Ordinal encoding

The 20 ordinal features become ordered integer codes (0 = lowest band),
using the orderings in config. Two examples below, raw label beside its code.

In [4]:
for column in ["study_consistency", "attendance_percentage"]:
    order = config.DS1_ORDINAL_FEATURES[column]
    code_map = {category: rank for rank, category in enumerate(order)}
    preview = pd.DataFrame({column: raw[column].head(6)})
    preview[f"{column}_code"] = preview[column].map(code_map)
    print(f"Ordering for {column}: {order}")
    display(preview)

Ordering for study_consistency: ['Rarely', 'Sometimes', 'Mostly consistent']


,study_consistency,study_consistency_code
0,Rarely,0
1,Sometimes,1
2,Mostly consistent,2
3,Sometimes,1
4,Sometimes,1
5,Mostly consistent,2


Ordering for attendance_percentage: ['Less than 50%', '50% – 65%', '66% – 75%', '76% – 85%', 'Above 85%']


,attendance_percentage,attendance_percentage_code
0,Above 85%,4
1,76% – 85%,3
2,Less than 50%,0
3,Above 85%,4
4,76% – 85%,3
5,Above 85%,4


## Nominal one-hot

The 7 nominal features have no order, so they get one-hot encoded. The number
of new columns is just the sum of their category counts.

In [5]:
nominal_cardinality = raw[config.DS1_NOMINAL_FEATURES].nunique().rename("n_categories")
display(nominal_cardinality.to_frame())
logger.info("Expected one-hot columns: %d", int(nominal_cardinality.sum()))

,n_categories
year_class,6
program_stream,7
main_distractor,5
skills_developing,3
career_interest,7
strongest_asset,4
internal_barrier,4


2026-07-03 18:32:01,566 | INFO | Expected one-hot columns: 36


## Fit and split

`prepare_primary_data` does the real work: a stratified train/test split on
the risk target, fit the preprocessor on the training split only, then
transform both and save everything. Fitting on train only is the whole point,
it keeps test information out of the medians, the scaler, and the encoder.

In [6]:
primary = prepare_primary_data(save=True)
x_train, x_test = primary["X_train"], primary["X_test"]
y_train, y_test = primary["y_train"], primary["y_test"]
preprocessor = primary["preprocessor"]

logger.info("Train matrix: %s | Test matrix: %s", x_train.shape, x_test.shape)
logger.info("Total engineered features: %d", len(preprocessor.feature_names_))
x_train.head()

2026-07-03 18:32:01,572 | INFO | Loaded behavioral_analytics: 1200 rows x 36 columns from behavioral_analytics.csv


2026-07-03 18:32:01,572 | INFO | behavioral_analytics passed schema validation.


2026-07-03 18:32:01,581 | INFO | Fitted BehavioralPreprocessor: 26 continuous + 36 one-hot = 62 features


2026-07-03 18:32:01,590 | INFO | Primary data prepared: train=960, test=240, features=62


2026-07-03 18:32:01,591 | INFO | Saved preprocessor -> /Users/agrim/Summer_Projects/EduSense/models/preprocessor.pkl


2026-07-03 18:32:01,611 | INFO | Wrote primary_train.csv and primary_test.csv


2026-07-03 18:32:01,612 | INFO | Train matrix: (960, 62) | Test matrix: (240, 62)


2026-07-03 18:32:01,612 | INFO | Total engineered features: 62


,age,daily_productivity,energy_level,stress_level,routine_rating,cgpa_category,attendance_percentage,academic_satisfaction,study_hours_daily,revision_frequency,...,career_interest_Other,career_interest_Software Developer,"strongest_asset_Creative/Design Skills (Innovation, UI/UX, Content)","strongest_asset_Management/Execution (Planning, Organizing, Discipline)","strongest_asset_Soft Skills (Communication, Leadership, Teamwork)","strongest_asset_Technical/Hard Skills (Coding, Math, Logic)",internal_barrier_Difficulty with Focus / Concentration,internal_barrier_Lack of Consistency or Determination (Difficulty sticking to a plan),internal_barrier_Poor Time Management / Over-scheduling,internal_barrier_Procrastination / Low Motivation
639,0.072160,1.977638,-1.104624,0.714013,-1.410466,1.631016,0.604626,-1.139202,1.688283,2.066844,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
313,-1.295081,0.261062,-0.210493,1.560250,-1.410466,1.631016,1.390067,1.517399,0.274015,-1.315264,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
432,0.072160,1.119350,-1.104624,1.560250,-1.410466,1.631016,-0.966256,-1.139202,0.274015,0.939475,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
94,2.350895,-1.455513,0.683637,0.714013,1.166218,-1.461365,-0.966256,-1.139202,0.274015,-0.187895,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
231,0.527907,-0.597225,1.577768,-0.132225,-1.410466,0.600223,0.604626,-0.253669,1.688283,2.066844,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


## Sanity checks

Three things I want to see before trusting the output: the split kept the
class balance, the continuous block is standardised (~0 mean, ~1 std) on
train, and there are no leftover nulls.

In [7]:
balance = pd.DataFrame(
    {
        "overall": raw[config.DS1_TARGET].map(config.RISK_LABEL_MAP).value_counts(normalize=True),
        "train": y_train.value_counts(normalize=True),
        "test": y_test.value_counts(normalize=True),
    }
).round(3)
balance.index = [config.RISK_LABEL_INVERSE[i] for i in balance.index]
print("Class balance (proportion) - stratification preserves it:")
display(balance)

continuous = preprocessor.continuous_features
scale_check = pd.DataFrame(
    {"mean": x_train[continuous].mean().round(3), "std": x_train[continuous].std().round(3)}
)
print("Continuous block on training data (should be ~0 mean, ~1 std):")
display(scale_check.head(8))

logger.info("Null values remaining in train matrix: %d", int(x_train.isnull().sum().sum()))
logger.info("Null values remaining in test matrix: %d", int(x_test.isnull().sum().sum()))

Class balance (proportion) - stratification preserves it:


,overall,train,test
Moderate Risk,0.477,0.477,0.475
High Risk,0.377,0.376,0.379
Low Risk,0.147,0.147,0.146


Continuous block on training data (should be ~0 mean, ~1 std):


,mean,std
age,0.0,1.001
daily_productivity,0.0,1.001
energy_level,0.0,1.001
stress_level,-0.0,1.001
routine_rating,-0.0,1.001
cgpa_category,0.0,1.001
attendance_percentage,0.0,1.001
academic_satisfaction,-0.0,1.001


2026-07-03 18:32:01,622 | INFO | Null values remaining in train matrix: 0


2026-07-03 18:32:01,622 | INFO | Null values remaining in test matrix: 0


## Regression data (DS3)

The score model trains on DS3, so split its six predictors and the exam-score
target 8000/2000. Scaling happens inside the model pipelines later, not here.

In [8]:
regression = prepare_regression_data(save=True)
logger.info(
    "Regression train: %s | test: %s",
    regression["X_train"].shape,
    regression["X_test"].shape,
)
print("DS3 features:", config.DS3_FEATURES)
print("\nExam-score target summary:")
display(regression["y_train"].describe().round(2).to_frame(config.DS3_TARGET))

2026-07-03 18:32:01,627 | INFO | Loaded performance_prediction: 10000 rows x 8 columns from performance_prediction.csv


2026-07-03 18:32:01,627 | INFO | performance_prediction passed schema validation.


2026-07-03 18:32:01,628 | INFO | Regression data prepared from DS3: train=8000, test=2000, features=6


2026-07-03 18:32:01,634 | INFO | Wrote regression_train.csv and regression_test.csv


2026-07-03 18:32:01,635 | INFO | Regression train: (8000, 6) | test: (2000, 6)


DS3 features: ['study_hours', 'attendance', 'sleep_hours', 'internet_usage', 'assignments_completed', 'previous_score']

Exam-score target summary:


,exam_score
count,8000.00
mean,86.76
std,15.03
min,26.67
25%,76.73
50%,92.18
75%,100.00
max,100.00


## Cross-dataset validation sets

DS2 (control) and DS4 (holdout) get cut down to the features they share with
DS3 and will be considered in further steops. DS2 is de-duplicated first. They share different
subsets, which is exactly why the cross-validator handles each one on its own.

In [9]:
val_ds2 = prepare_validation_set("ds2", save=True)
val_ds4 = prepare_validation_set("ds4", save=True)

print("DS2 (control) shared features:", [c for c in val_ds2.columns if c != "exam_score"])
display(val_ds2.head(4))
print("DS4 (holdout) shared features:", [c for c in val_ds4.columns if c != "exam_score"])
display(val_ds4.head(4))

2026-07-03 18:32:01,642 | INFO | Loaded zenodo_merged: 14003 rows x 16 columns from zenodo_merged.csv


2026-07-03 18:32:01,642 | INFO | zenodo_merged passed schema validation.


2026-07-03 18:32:01,644 | INFO | DS2 de-duplicated: 14003 -> 12469 rows


2026-07-03 18:32:01,644 | INFO | Validation set ds2 prepared: 12469 rows, shared features ['study_hours', 'attendance', 'assignment_completion']


2026-07-03 18:32:01,649 | INFO | Wrote val_ds2.csv


...

2026-07-03 18:32:01,657 | INFO | Wrote val_ds4.csv


DS2 (control) shared features: ['study_hours', 'attendance', 'assignment_completion']


,study_hours,attendance,assignment_completion,exam_score
0,19,64,59,40
1,19,64,90,66
2,19,64,67,99
3,19,64,59,40


DS4 (holdout) shared features: ['study_hours', 'attendance', 'previous_score', 'sleep_hours']


,study_hours,attendance,previous_score,sleep_hours,exam_score
0,23,84,73,7,67
1,19,64,59,8,61
2,24,98,91,7,74
3,29,89,98,8,71


## Takeaways

- Every learned statistic is fit on the training split only, so the test
  split and the holdout stay unseen.
- DS1 expands to 62 features with no leftover nulls, and the stratified split
  keeps the Low/Moderate/High balance in both splits.
- Procrastination is derived in one place (preprocessing plus the config
  rules), so EDA, modelling, and inference all compute it the same way.
- The regression and validation data come out in canonical feature space,
  ready for the steps that follow.